# Agent Eval Harness

This notebook runs the benchmark harness in `tests/evals/`.

Before running:
- copy `.env.example` to `.env`
- fill in the API keys for the providers you want to compare
- edit `MODEL_SPECS` below as needed
- judge defaults centrally to `claude-opus-4-6`


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "miso").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests.evals import DEFAULT_RUBRIC_WEIGHTS, list_eval_cases, render_suite_summary, run_benchmark_suite


In [ ]:
MODEL_SPECS = [
    {
        "provider": "openai",
        "model": "gpt-5",
        "label": "gpt-5",
        "api_key_env": "OPENAI_API_KEY",
    },
    {
        "provider": "anthropic",
        "model": "claude-sonnet-4-6",
        "label": "claude-sonnet-4-6",
        "api_key_env": "ANTHROPIC_API_KEY",
    },
    {
        "provider": "gemini",
        "model": "gemini-2.5-pro",
        "label": "gemini-2.5-pro",
        "api_key_env": "GEMINI_API_KEY",
    },
]

SELECTED_CASE_IDS = ["repo_trace", "fixture_debug", "multi_file_plan"]
RUBRIC_WEIGHTS = dict(DEFAULT_RUBRIC_WEIGHTS)
SCORE_WEIGHTS = {"rule": 0.4, "judge": 0.6}
MAX_ITERATIONS = 8


In [ ]:
AVAILABLE_CASES = list_eval_cases(REPO_ROOT)
for case in AVAILABLE_CASES:
    print(f"{case.id}: {case.title}")


In [ ]:
result = run_benchmark_suite(
    repo_root=REPO_ROOT,
    model_specs=MODEL_SPECS,
    selected_case_ids=SELECTED_CASE_IDS,
    rubric_weights=RUBRIC_WEIGHTS,
    score_weights=SCORE_WEIGHTS,
    max_iterations=MAX_ITERATIONS,
)

print(render_suite_summary(result))
result["leaderboard_rows"]


In [ ]:
from pathlib import Path

summary_path = Path(result["summary_path"])
leaderboard_path = Path(result["leaderboard_path"])

print(f"Summary JSON: {summary_path}")
print(f"Leaderboard CSV: {leaderboard_path}")
print()
print(leaderboard_path.read_text(encoding='utf-8'))
